In [1]:
import os 
import datetime
import pandas as pd
from time import sleep
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider
import asyncio
from google.genai.errors import ServerError

In [3]:
base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"

#--- load api key
load_dotenv(override=True)
API_KEY = os.environ['PAID_GEMMA_API_KEY']

#--- initialize the agent
SYSTEM_PROMPT = """
    You are a helpful assistant that converts the url link of a news article to sentence like topic. As an input you will get bulk of links as string.
    Your task is to return two or maximum three sentences that are going to give some type of description about what happened to gold market or gold prices.
    While generating the answer make sure that:
    -Try to approximate the situation as close as you can.
    -Answer only the topic, do not add extra discussion. 
    -Do not format the answer.  
    Make sure that you capture all the necessary information and the details about the text that is provided you.
"""

provider = GoogleProvider(api_key=API_KEY)
model = GoogleModel("gemini-2.0-flash", provider=provider)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT, output_type=str, output_retries=5)

#---main logic
start_date = datetime.date(2016, 6, 27)
end_date = datetime.date(2016, 10, 28)
current_date = start_date
results = []
missing_dates = []


while current_date <= end_date:
    # Prepare batch of 2 days
    batch_dates = [current_date]
    next_date = current_date + datetime.timedelta(days=1)
    if next_date <= end_date:
        batch_dates.append(next_date)

    print(f"Processing batch for dates: {[d.strftime('%Y-%m-%d') for d in batch_dates]}")

    url_bulk = ""
    missing_in_batch = []

    for date in batch_dates:
        date_str = date.strftime("%Y%m%d")
        file_name = f"{date_str}_gold_filtered.csv"
        full_path = os.path.join(base_path, file_name)

        try:
            df = pd.read_csv(full_path)
            for url in df['SOURCEURL']:
                url_bulk += f" {url}"
        except FileNotFoundError:
            print(f"File not found: {full_path}")
            missing_in_batch.append(date)
        except pd.errors.EmptyDataError:
            print(f"Empty CSV: {full_path}")
            missing_in_batch.append(date)
        except Exception as e:
            print(f"Error processing {full_path}: {e}")
            missing_in_batch.append(date)

    if url_bulk.strip():
        while True:
            try:
                answer = await agent.run(url_bulk)
                break  # success → exit retry loop
            except ServerError as e:
                if getattr(e, "status_code", None) == 503 or "model is overloaded" in str(e).lower():
                    print(f"Model overloaded. Waiting 2 minutes before retrying the same batch...")
                    await asyncio.sleep(120)
                else:
                    raise

        text = answer.output

        for date in batch_dates:
            if date not in missing_in_batch:
                results.append({
                    "date": date.strftime("%Y%m%d"),
                    "text": text
                })
            else:
                missing_dates.append(date)
    else:
        missing_dates.extend(missing_in_batch)

    sleep(6)
    current_date += datetime.timedelta(days=2)


df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data3_1.csv"
df.to_csv(path, index=False)

print(f"============= MISSING DATA FOR TOTAL OF {len(missing_dates)} DATES : =============")
print(missing_dates)

Processing batch for dates: ['2016-06-27', '2016-06-28']
Processing batch for dates: ['2016-06-29', '2016-06-30']
Processing batch for dates: ['2016-07-01', '2016-07-02']
Processing batch for dates: ['2016-07-03', '2016-07-04']
Processing batch for dates: ['2016-07-05', '2016-07-06']
Processing batch for dates: ['2016-07-07', '2016-07-08']
Processing batch for dates: ['2016-07-09', '2016-07-10']
Processing batch for dates: ['2016-07-11', '2016-07-12']
Processing batch for dates: ['2016-07-13', '2016-07-14']
Processing batch for dates: ['2016-07-15', '2016-07-16']
Processing batch for dates: ['2016-07-17', '2016-07-18']
Processing batch for dates: ['2016-07-19', '2016-07-20']
Processing batch for dates: ['2016-07-21', '2016-07-22']
Processing batch for dates: ['2016-07-23', '2016-07-24']
Processing batch for dates: ['2016-07-25', '2016-07-26']
Processing batch for dates: ['2016-07-27', '2016-07-28']
Processing batch for dates: ['2016-07-29', '2016-07-30']
Processing batch for dates: ['2

In [ ]:
df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data5.csv"
df.to_csv(path, index=False)

       date                                               text
0  20170510  China's private investor gold demand is surgin...
1  20170511  China's private investor gold demand is surgin...
2  20170512  Transition Metals Corp closes private placemen...
3  20170513  Transition Metals Corp closes private placemen...
4  20170514  Customs officials in Bangladesh seized 300 kg ...
